Un departamento de Riesgos necesita evaluar diariamente la exposición de un portafolio de equities mexicanos. El problema es que los datos históricos crudos que se reciben a veces vienen con huecos por dias feriados locales no sincronizados, errores de transmisión o con precios en cero por falta de liquidez intradiaria. Si se calcula la volatilidad o el riesgo con estos errores, las métricas se disparan de forma artificial o rompen los sistemas de pricing en la cadena. 

Desarrollar una herramienta que automatice la limpieza y calcule el riesgo de cola.

**Requerimientos técnicos:**
1. **Adquisición:** Escribe un script que descargue los precios de cierre ajustado de los últimos 2 años para los siguientes activos: WALMEX.MX, CEMEXCPO.MX, GFNORTEO.MX, GMEXICOB.MX y GCARSOA.MX (puedes usar la librería yfinance para simular nuestra base de datos).

2. **Preprocesamiento:** Construye una función de limpieza que reciba este DataFrame e:
    - Identifique y maneje valores nulos (NaN) o precios marcados como 0.0. (Nota: En los docstrings de tu función, debes justificar brevemente tu decisión de imputación: ¿hiciste forward-fill, backward-fill, interpolación o drop? ¿Por qué?)
    - Calcule los retornos logarítmicos diarios, ya que son aditivos a través del tiempo.
3. **Agregación:** Asumiendo un portafolio equiponderado (pesos iguales), calcula la serie de tiempo de los retornos diarios del portafolio completo.

4. **Cálculo de Riesgo:** Crea una función que tome los retornos del portafolio y calcule:
- Value at Risk (VaR) Histórico al nivel de confianza del $99\%$.
- Expected Shortfall (CVaR) Histórico al nivel de confianza del $99\%$. (Recuerda que el CVaR es la expectativa condicional $\mathbb{E}[L \vert{} L > \text{VaR}_\alpha]$).

In [ ]:
# Importar las librerias necesarias
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

In [21]:
# Definimos el universo de activos del portafolio y la temporalidad
universo = ["WALMEX.MX", "CEMEXCPO.MX", "GFNORTEO.MX", "GMEXICOB.MX", "GCARSOA1.MX"]
inicio = "2024-01-01"
fin = datetime.today().strftime("%Y-%m-%d")

data = yf.download(tickers=universo, start=inicio, end=fin, interval="1D")["Close"]

[*********************100%***********************]  5 of 5 completed


In [22]:
# Como la serie de retornos creada se usara para modelos predictivos, debemos de prevenir look ahead bias, para esto usaremos forward fill ya que toma al último precio conocido cuando hay valores faltantes.
def limpieza_datos(data):
    """
    Recibe como entrada un dataframe de precios de activos que pueden contener valores faltantes o 0.00.
    Limpia los valores faltantes
    Genera como salida una serie de retornos logarítmicos.
    """
    # 1. Verificamos si hay valores nulos o en 0.00
    data = data.replace(0.0, np.nan).ffill()

    # 3. Una vez el df esta limpio, calculamos los retornos logaritmicos
    return np.log(data / data.shift(1)).dropna()

In [23]:
log_returns = limpieza_datos(data=data)

In [24]:
# Creamos la función para el portafolio de pesos iguales
def EW_portfolio_log_returns(log_returns):
    """
    Recibe como entrada un dataframe de retornos logarítmicos de n activos
    Genera como salida un dataframe de rendimientos logarítmicos diarios para un portafolio de pesos iguales.
    """
    # 1. Generamos los pesos para el portafolio de pesos iguales
    weights = np.array([1 / len(log_returns.columns)] * len(log_returns.columns))

    # 2. Calculamos los retornos logaritmocos para el portafolio
    return log_returns.dot(weights)

In [25]:
EWpf_log_returns = EW_portfolio_log_returns(log_returns)

In [ ]:
# Creamos las funciones para VaR y CVAR
def VAR_historic(portfolio_log_returns, start_price, confidence_interval=99):
    """
    Recibe como entrada una serie temporal de los retornos logarítmicos de un portafolio, un monto de inversión inicial y un intervalo de confianza, por default es 99. 
    Genera como salida el VaR histórico.
    """
    # 1. Encontrar el percentil de los retornos logarítmicos
    log_percentile = np.percentile(portfolio_log_returns, 100 - confidence_interval)

    # 2. Convertir de retornos logarítmicos a simples
    simple_percentile = np.exp(log_percentile) - 1 

    # 3. Calcular el VaR historico
    return - simple_percentile * start_price

- **data:** Representa a los precios diarios de los activos que estan dentro del portafolio
- **log_returns:** Usa la función limpieza_datos para limpiar el df y generar una serie de retornos logarítmicos
- **EWpf_log_returns:** Usa la función EW_portfolio_log_returns para generar una serie de retornos logarítimicos para un portafolio de pesos iguales


In [27]:
EWpf_VAR_historic = VAR_historic(EWpf_log_returns, 100)
EWpf_VAR_historic

np.float64(3.269024612138416)

In [ ]:
def CVAR(portfolio_log_returns, start_price, confidence_interval):
    """
    Recibe como entrada una serie de retornos logaritmicos de un portafolio, un monto de inversión inicial y un intervalo de confianza.
    Genera como salida el CVAR
    """
    # 1. Ordenar los retornos logarítmicos del portafolio de menor a mayor
    sorted_log_returns = np.sort(portfolio_log_returns)

    # 2. Encontrar el percentil logaritmico con el que calcularemos nuestro CVAR
    log_percentile = np.percentile(portfolio_log_returns, 100 - confidence_interval)

    # 3. Filtrar los retornos que sean igual o menor que el percentil para obtener la cola
    tail_losses = sorted_log_returns[sorted_log_returns <= log_percentile]

    # 4. Calcular la media aritmetica de la cola
    log_cvar = np.mean(tail_losses)

    #5. Transformar el CVAR logarítmico a simple
    simple_cvar = - (np.exp(log_cvar) - 1)

    # 6. Multiplicar por nuestra inversión inicial
    return simple_cvar * start_price

In [29]:
CVAR(EWpf_log_returns, 100, 99)

np.float64(4.405688208798186)